In [2]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor
from rl_mind.core import Action, Actor
from rl_mind.env import VecEnv
from rl_mind.data import Episode, Transitions
from rl_mind.collectors import EpisodeCollector, RolloutCollector, TransitionCollector
from rl_mind import RenderWidget, DecimateTrimmer
from rl_mind.notebook import silence_known_warnings



In [3]:
silence_known_warnings()

In [4]:
env = VecEnv("LunarLander-v3", num_envs=10, seed=1, continuous=True)


## A random actor for testing purpose 

In [ ]:
class RandomActor(Actor[Action]):
    def __init__(self,action_dim: int):
        super().__init__()
        self.action_dim = action_dim
    def forward(self,obs: Tensor) -> Action:
        return Action(value=torch.empty(obs.shape[0],self.action_dim).uniform_(-1,1))
print(env.action_dim) # action dim = 2 because continuous see documentation
actor = RandomActor(env.action_dim)

2


Main engine (first value): 

Below 0, the engine is off.
From 0 to 1, it fires, with throttle going from 50% to 100%.

Lateral engines (second value):

From -1 to -0.5, the left engine fires.
From -0.5 to 0.5, both are off.
From 0.5 to 1, the right engine fires.

In [8]:
obs = env.reset()
total = 0.0
for t in range(300):
    step = env.step(actor(obs).value) # actor(obs).value extracts the action tensor
    total += step.reward.item()
    obs = step.obs
print("return : ",total)

return :  -358.645637255162


## Visualization 

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video

genv = RecordVideo(
    gym.make("LunarLander-v3", continuous=True, render_mode="rgb_array"),
    video_folder="videos", episode_trigger=lambda i: True,
)
obs, _ = genv.reset(seed=1)
done = False
while not done:
    obs, r, term, trunc, _ = genv.step(genv.action_space.sample())  # random action
    done = term or trunc
genv.close()

Video("../videos/rl-video-episode-random.mp4", embed=True)